In [ ]:
import os
import gc
import re
import json
import math
import random
import logging
import warnings
import threading

from pathlib import Path
from dataclasses import dataclass, field
from contextlib import nullcontext
from typing import List, Tuple, Dict

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings("ignore")

MADLAB_PREFIX = "<2en> "

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")


# ---------------------------------------------------------------------------
# Utility helpers
# ---------------------------------------------------------------------------

def _cuda_bf16_supported(device: torch.device = None) -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        if device is not None and device.type == "cuda":
            idx = device.index or 0
            with torch.cuda.device(idx):
                return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
        return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except Exception:
        return False


def _bf16_ctx(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda" and _cuda_bf16_supported(device):
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


def _num_gpus() -> int:
    return torch.cuda.device_count() if torch.cuda.is_available() else 0


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

@dataclass
class EnsembleMBRConfig:
    test_data_path: str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    output_dir: str = "/kaggle/working/"

    model_paths: List[str] = field(default_factory=lambda: [
        "/kaggle/input/datasets/ravikshdikola/madlab-akkadian-retrained-1st-newold-merged",
        "/kaggle/input/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-ciphermix-2ep",
        "/kaggle/input/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-merged",
        "/kaggle/input/datasets/ravikshdikola/dpc2-exp2/google_madlad400_3b_mt_finetuned_corrected",
        "/kaggle/input/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-newmix",
        "/kaggle/input/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-3mix-new/model"
        
    ])

    max_input_length: int = 576
    max_new_tokens: int   = 576

    batch_size: int = 4
    num_workers: int = 2
    num_buckets: int = 6

    num_beam_cands: int   = 12
    num_beams: int        = 12
    length_penalty: float = 1.6
    early_stopping: bool  = True
    repetition_penalty: float = 1.5

    num_sample_cands: int  = 4
    mbr_top_p: float       = 0.94
    mbr_temperature: float = 0.80
    mbr_pool_cap: int      = 80

    use_mixed_precision: bool     = True
    use_better_transformer: bool  = True
    use_bucket_batching: bool     = True
    use_adaptive_beams: bool      = True
    aggressive_postprocessing: bool = True
    checkpoint_freq: int          = 200

    # ── Dual-GPU ────────────────────────────────────────────────────────────
    # When True and 2+ GPUs are available, models are split across GPUs and
    # run in parallel threads — roughly halving wall-clock inference time.
    use_multi_gpu: bool = True

    def __post_init__(self):
        n_gpus = _num_gpus()

        # Primary device (used for single-GPU fallback and MBR)
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # Build per-model device assignment
        # Layout for 4 models on 2 GPUs:
        #   cuda:0 → models 0, 1
        #   cuda:1 → models 2, 3
        if self.use_multi_gpu and n_gpus >= 2:
            self.model_devices = [
                torch.device(f"cuda:{i % n_gpus}")
                for i in range(len(self.model_paths))
            ]
        else:
            self.model_devices = [self.device] * len(self.model_paths)

        Path(self.output_dir).mkdir(exist_ok=True, parents=True)

        if not torch.cuda.is_available():
            self.use_mixed_precision    = False
            self.use_better_transformer = False

        self.use_bf16_amp = bool(
            self.use_mixed_precision
            and self.device.type == "cuda"
            and _cuda_bf16_supported(self.device)
        )

        assert self.num_beams >= self.num_beam_cands
        assert len(self.model_paths) >= 1

        n_gpus = _num_gpus()
        multi  = self.use_multi_gpu and n_gpus >= 2
        print(
            f"[Config] GPUs detected: {n_gpus} | "
            f"Multi-GPU: {'ON ✓' if multi else 'OFF'}"
        )
        for i, (p, d) in enumerate(zip(self.model_paths, self.model_devices)):
            print(f"  Model {i+1} → {d}  |  {Path(p).name}")


# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

def setup_logging(output_dir: str) -> logging.Logger:
    Path(output_dir).mkdir(exist_ok=True, parents=True)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(output_dir) / "ensemble_mbr.log"),
        ],
    )
    return logging.getLogger("ensemble_mbr")


# ---------------------------------------------------------------------------
# Text normalisation helpers  (unchanged)
# ---------------------------------------------------------------------------

_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a": "á", "e": "é", "i": "í", "u": "ú",
                         "A": "Á", "E": "É", "I": "Í", "U": "Ú"})
_GRAVE = str.maketrans({"a": "à", "e": "è", "i": "ì", "u": "ù",
                         "A": "À", "E": "È", "I": "Ì", "U": "Ù"})


def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


_ALLOWED_FRACS = [
    (1/6, "0.16666"), (1/4, "0.25"), (1/3, "0.33333"),
    (1/2, "0.5"), (2/3, "0.66666"), (3/4, "0.75"), (5/6, "0.83333"),
]
_FRAC_TOL  = 2e-3
_FLOAT_RE  = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")


def _canon_decimal(x: float) -> str:
    ip   = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= _FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")


_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I,
)


def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ": "h", "Ḫ": "H", "ʾ": "",
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
    "—": "-", "–": "-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")
_PN_RE        = re.compile(r"\bPN\b")
_KUBABBAR_RE  = re.compile(r"KÙ\.B\.")

_EXACT_FRAC_RE  = re.compile(
    r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5"
)
_EXACT_FRAC_MAP = {
    "0.8333": "⅚", "0.6666": "⅔", "0.3333": "⅓", "0.1666": "⅙",
    "0.625": "⅝", "0.75": "¾", "0.25": "¼", "0.5": "½",
}


def _frac_repl(m: re.Match) -> str:
    return _EXACT_FRAC_MAP[m.group(0)]


# ---------------------------------------------------------------------------
# Preprocessor
# ---------------------------------------------------------------------------

class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)
        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        ser = ser.str.replace(
            _FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True
        )
        ser = ser.str.replace(r"\s+", " ", regex=True).str.strip()
        return ser.tolist()


# ---------------------------------------------------------------------------
# Postprocessor
# ---------------------------------------------------------------------------

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)",
    re.I,
)
_BARE_GRAM_RE    = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE    = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")
_MONTH_RE        = re.compile(
    r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I
)
_ROMAN2INT = {
    "I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6,
    "VII": 7, "VIII": 8, "IX": 9, "X": 10, "XI": 11, "XII": 12,
}
_REPEAT_WORD_RE  = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE  = re.compile(r"\s+([.,:])")
_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')
_COMMODITY_RE    = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL  = {
    "gold": "pašallum gold", "tax": "šadduātum tax", "textiles": "kutānum textiles",
}


def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]


_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I),      '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I),      '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]
_SLASH_ALT_RE   = re.compile(r'(?<!\d)\s*/\s*(?!\d)\S+')
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE   = re.compile(r'(?:<gap>\s*){2,}')
_WS_RE          = re.compile(r"\s+")


def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)
        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)
        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)
        s = s.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        s = s.str.replace(
            _FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True
        )
        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)
        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        s = s.str.replace(_SLASH_ALT_RE, "", regex=True)
        s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)
        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)
        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)
        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n - 1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)
        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()
        return s.tolist()


# ---------------------------------------------------------------------------
# Dataset & Sampler
# ---------------------------------------------------------------------------

class AkkadianDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        preprocessor: OptimizedPreprocessor,
        logger: logging.Logger,
    ):
        self.sample_ids  = df["id"].tolist()
        self.input_texts = preprocessor.preprocess_batch(
            df["transliteration"].tolist()
        )
        logger.info(f"Dataset: {len(self.sample_ids)} samples")

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        return self.sample_ids[idx], self.input_texts[idx]


class BucketBatchSampler(Sampler):
    def __init__(
        self,
        dataset: AkkadianDataset,
        batch_size: int,
        num_buckets: int,
        logger: logging.Logger,
        shuffle: bool = False,
    ):
        self.batch_size = batch_size
        self.shuffle    = shuffle

        lengths    = [len(t.split()) for _, t in dataset]
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])

        bsize = max(1, len(sorted_idx) // max(1, num_buckets))
        self.buckets = [
            sorted_idx[
                i * bsize: None if i == num_buckets - 1 else (i + 1) * bsize
            ]
            for i in range(num_buckets)
        ]
        for i, b in enumerate(self.buckets):
            if b:
                bl = [lengths[x] for x in b]
                logger.info(
                    f"  Bucket {i}: {len(b)} samples, "
                    f"len [{min(bl)}, {max(bl)}]"
                )

    def __iter__(self):
        for bucket in self.buckets:
            b = list(bucket)
            if self.shuffle:
                random.shuffle(b)
            for i in range(0, len(b), self.batch_size):
                yield b[i: i + self.batch_size]

    def __len__(self):
        return sum(math.ceil(len(b) / self.batch_size) for b in self.buckets)


# ---------------------------------------------------------------------------
# Model wrapper  ── now device-aware
# ---------------------------------------------------------------------------

class ModelWrapper:
    """
    Each ModelWrapper is pinned to a specific device (cuda:0 or cuda:1).
    The device is passed in from EnsembleMBRConfig.model_devices[i].
    """

    def __init__(
        self,
        model_path: str,
        cfg: EnsembleMBRConfig,
        logger: logging.Logger,
        label: str,
        device: torch.device,          # ← explicit per-model device
    ):
        self.cfg    = cfg
        self.logger = logger
        self.label  = label
        self.device = device           # ← stored separately from cfg.device

        logger.info(f"[{label}] Loading → {device}  |  {model_path}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = (
            AutoModelForSeq2SeqLM.from_pretrained(
                model_path, torch_dtype=torch.float16
            )
            .to(device)
            .eval()
        )

        if device.type == "cuda":
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass

        n = sum(p.numel() for p in self.model.parameters())
        logger.info(f"[{label}] {n:,} parameters on {device}")

        if device.type == "cuda":
            used = torch.cuda.memory_allocated(device) / 1e9
            logger.info(f"[{label}] GPU mem used: {used:.2f} GB")

        if cfg.use_better_transformer and device.type == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                logger.info(f"[{label}] BetterTransformer applied")
            except Exception as e:
                logger.warning(f"[{label}] BetterTransformer skipped: {e}")

    # ------------------------------------------------------------------

    def collate(self, batch_samples):
        ids   = [s[0] for s in batch_samples]
        texts = [MADLAB_PREFIX + s[1] for s in batch_samples]
        enc   = self.tokenizer(
            texts,
            max_length=self.cfg.max_input_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        return ids, enc

    # ------------------------------------------------------------------

    def generate_candidates(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        beam_size: int,
    ) -> List[List[str]]:
        cfg    = self.cfg
        device = self.device
        B      = input_ids.shape[0]

        # Use per-model device for autocast
        bf16_on = (
            cfg.use_mixed_precision
            and device.type == "cuda"
            and _cuda_bf16_supported(device)
        )
        ctx = (
            torch.autocast(device_type="cuda", dtype=torch.bfloat16)
            if bf16_on else nullcontext()
        )

        common_kwargs = dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=cfg.max_new_tokens,
            repetition_penalty=cfg.repetition_penalty,
            use_cache=True,
        )

        with ctx:
            nb       = max(beam_size, cfg.num_beam_cands)
            beam_out = self.model.generate(
                **common_kwargs,
                do_sample=False,
                num_beams=nb,
                num_return_sequences=cfg.num_beam_cands,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
            )
            beam_texts = self.tokenizer.batch_decode(
                beam_out, skip_special_tokens=True
            )

            samp_texts = []
            if cfg.num_sample_cands > 0:
                samp_out = self.model.generate(
                    **common_kwargs,
                    do_sample=True,
                    num_beams=1,
                    top_p=cfg.mbr_top_p,
                    temperature=cfg.mbr_temperature,
                    num_return_sequences=cfg.num_sample_cands,
                )
                samp_texts = self.tokenizer.batch_decode(
                    samp_out, skip_special_tokens=True
                )

        Rb, Rs = cfg.num_beam_cands, cfg.num_sample_cands
        pools  = []
        for i in range(B):
            p = list(beam_texts[i * Rb: (i + 1) * Rb])
            if Rs > 0:
                p += list(samp_texts[i * Rs: (i + 1) * Rs])
            pools.append(p)
        return pools

    # ------------------------------------------------------------------

    def unload(self):
        label  = self.label
        device = self.device
        try:
            from optimum.bettertransformer import BetterTransformer
            self.model = BetterTransformer.reverse(self.model)
        except Exception:
            pass
        del self.model
        del self.tokenizer
        self.model     = None
        self.tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize(device)
            free = (
                torch.cuda.get_device_properties(device).total_memory
                - torch.cuda.memory_allocated(device)
            ) / 1e9
            self.logger.info(f"[{label}] Unloaded {device}. Free: {free:.2f} GB")



# ---------------------------------------------------------------------------
# MBR selector  ── contest-aligned scoring
# ---------------------------------------------------------------------------

class MBRSelector:
    def __init__(self, pool_cap: int = 32):
        self._chrf    = sacrebleu.metrics.CHRF(word_order=2)
        self._bleu    = sacrebleu.metrics.BLEU(effective_order=True)
        self.pool_cap = pool_cap

    def _score(self, a: str, b: str) -> float:
        a, b = (a or "").strip(), (b or "").strip()
        if not a or not b:
            return 0.0
        chrf = float(self._chrf.sentence_score(a, [b]).score)
        bleu = float(self._bleu.sentence_score(a, [b]).score)
        if chrf == 0.0 or bleu == 0.0:
            return 0.0
        return (chrf * bleu) ** 0.5

    @staticmethod
    def _dedup(xs: List[str]) -> List[str]:
        seen, out = set(), []
        for x in xs:
            x = str(x).strip()
            if x and x not in seen:
                out.append(x)
                seen.add(x)
        return out

    def pick(self, candidates: List[str]) -> str:
        cands = self._dedup(candidates)
        if self.pool_cap:
            cands = cands[: self.pool_cap]
        n = len(cands)
        if n == 0:
            return ""
        if n == 1:
            return cands[0]
        best_i, best_s = 0, -1e9
        for i in range(n):
            s = sum(self._score(cands[i], cands[j]) for j in range(n) if j != i)
            s /= max(1, n - 1)
            if s > best_s:
                best_s, best_i = s, i
        return cands[best_i]


  

# ---------------------------------------------------------------------------
# Main engine  ── dual-GPU parallel inference
# ---------------------------------------------------------------------------

class EnsembleMBREngine:
    def __init__(self, cfg: EnsembleMBRConfig, logger: logging.Logger):
        self.cfg           = cfg
        self.logger        = logger
        self.preprocessor  = OptimizedPreprocessor()
        self.postprocessor = VectorizedPostprocessor()
        self.mbr           = MBRSelector(pool_cap=cfg.mbr_pool_cap)

    # ------------------------------------------------------------------

    def _adaptive_beams(self, attn: torch.Tensor) -> int:
        if not self.cfg.use_adaptive_beams:
            return self.cfg.num_beams
        med   = float(attn.sum(dim=1).float().median().item())
        short = max(self.cfg.num_beam_cands, self.cfg.num_beams // 2)
        return short if med < 100 else self.cfg.num_beams

    def _build_dataloader(
        self, dataset: AkkadianDataset, wrapper: ModelWrapper
    ) -> DataLoader:
        if self.cfg.use_bucket_batching:
            sampler = BucketBatchSampler(
                dataset,
                self.cfg.batch_size,
                self.cfg.num_buckets,
                self.logger,
            )
            return DataLoader(
                dataset,
                batch_sampler=sampler,
                num_workers=self.cfg.num_workers,
                collate_fn=wrapper.collate,
                pin_memory=(self.cfg.device.type == "cuda"),
            )
        return DataLoader(
            dataset,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            num_workers=self.cfg.num_workers,
            collate_fn=wrapper.collate,
            pin_memory=(self.cfg.device.type == "cuda"),
        )

    # ------------------------------------------------------------------

    def _run_one_model(
        self,
        wrapper: ModelWrapper,
        dataset: AkkadianDataset,
        result_store: Dict,         # shared dict — thread-safe writes by key
        store_lock: threading.Lock,
    ) -> None:
        """
        Runs inference for a single model and stores results in result_store.
        Designed to run inside a thread so multiple models execute in parallel.
        """
        device    = wrapper.device
        dl        = self._build_dataloader(dataset, wrapper)
        local_map: Dict[str, List[str]] = {}

        with torch.inference_mode():
            for batch_ids, enc in tqdm(
                dl, desc=f"[{wrapper.label} | {device}]", position=wrapper.device.index
            ):
                input_ids = enc.input_ids.to(device, non_blocking=True)
                attn      = enc.attention_mask.to(device, non_blocking=True)
                beam_size = self._adaptive_beams(attn)

                try:
                    batch_pools = wrapper.generate_candidates(
                        input_ids, attn, beam_size
                    )
                    for sid, pool in zip(batch_ids, batch_pools):
                        local_map[str(sid)] = pool

                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        self.logger.error(
                            f"OOM [{wrapper.label}|{device}] — skipping batch"
                        )
                        torch.cuda.empty_cache()
                        for sid in batch_ids:
                            local_map.setdefault(str(sid), [])
                    else:
                        raise

                except Exception as e:
                    self.logger.error(f"[{wrapper.label}] batch error: {e}")
                    for sid in batch_ids:
                        local_map.setdefault(str(sid), [])

                torch.cuda.empty_cache()

        # Merge local results into shared store
        with store_lock:
            result_store[wrapper.label] = local_map

    # ------------------------------------------------------------------

    def _run_group_parallel(
        self,
        model_indices: List[int],
        dataset: AkkadianDataset,
    ) -> List[Dict[str, List[str]]]:
        """
        Load all models in `model_indices`, run them in parallel threads
        (one thread per model / GPU), then unload all.

        Returns list of pools dicts in the same order as model_indices.
        """
        cfg    = self.cfg
        logger = self.logger

        wrappers = []
        for i in model_indices:
            label   = f"MadLab-{i + 1}"
            device  = cfg.model_devices[i]
            wrapper = ModelWrapper(
                cfg.model_paths[i], cfg, logger, label, device
            )
            wrappers.append(wrapper)

        result_store: Dict[str, Dict] = {}
        store_lock = threading.Lock()
        threads    = []

        for wrapper in wrappers:
            t = threading.Thread(
                target=self._run_one_model,
                args=(wrapper, dataset, result_store, store_lock),
                name=wrapper.label,
                daemon=True,
            )
            threads.append(t)

        logger.info(
            f"Starting {len(threads)} parallel inference thread(s): "
            + ", ".join(w.label for w in wrappers)
        )
        for t in threads:
            t.start()
        for t in threads:
            t.join()

        # Unload all models in this group
        for wrapper in wrappers:
            wrapper.unload()
            del wrapper

        # Return pools in original order
        return [result_store.get(f"MadLab-{i + 1}", {}) for i in model_indices]

    # ------------------------------------------------------------------

    def run(self, test_df: pd.DataFrame) -> pd.DataFrame:
        cfg, logger = self.cfg, self.logger
        n_gpus      = _num_gpus()
        multi_gpu   = cfg.use_multi_gpu and n_gpus >= 2

        logger.info("=" * 60)
        logger.info(f"Ensemble MBR  —  {len(cfg.model_paths)} models")
        logger.info(f"  Multi-GPU    : {'ON ✓  (2 × T4 parallel)' if multi_gpu else 'OFF'}")
        logger.info(f"  GPUs         : {n_gpus}")
        logger.info(f"  prefix       : '{MADLAB_PREFIX}'")
        logger.info(f"  batch_size   : {cfg.batch_size}")
        logger.info(f"  num_beams    : {cfg.num_beams}")
        logger.info(f"  beam_cands   : {cfg.num_beam_cands}")
        logger.info(f"  sample_cands : {cfg.num_sample_cands}")
        logger.info(f"  MBR pool cap : {cfg.mbr_pool_cap}")
        for i, (p, d) in enumerate(zip(cfg.model_paths, cfg.model_devices)):
            logger.info(f"  [MadLab-{i+1}] {d} | {p}")
        logger.info("=" * 60)

        dataset    = AkkadianDataset(test_df, self.preprocessor, logger)
        sample_ids = [str(s) for s in dataset.sample_ids]
        all_pools: List[Dict[str, List[str]]] = []

        if multi_gpu:
            # ── Dual-GPU: run models 2 at a time, each on its own GPU ──────
            # Group 1: models 0,1  (cuda:0 + cuda:1 simultaneously)
            # Group 2: models 2,3  (cuda:0 + cuda:1 simultaneously)
            group_size = n_gpus
            for g_start in range(0, len(cfg.model_paths), group_size):
                g_end    = min(g_start + group_size, len(cfg.model_paths))
                indices  = list(range(g_start, g_end))
                labels   = [f"MadLab-{i+1}" for i in indices]
                logger.info(
                    f"── Parallel group: {labels} "
                    f"on {[str(cfg.model_devices[i]) for i in indices]} ──"
                )
                group_pools = self._run_group_parallel(indices, dataset)
                all_pools.extend(group_pools)

        else:
            # ── Single-GPU fallback: sequential ────────────────────────────
            for idx, model_path in enumerate(cfg.model_paths, start=1):
                label   = f"MadLab-{idx}"
                device  = cfg.model_devices[idx - 1]
                logger.info(f"Phase {idx}/{len(cfg.model_paths)} — [{label}] on {device}")
                wrapper = ModelWrapper(model_path, cfg, logger, label, device)
                result_store: Dict = {}
                lock = threading.Lock()
                self._run_one_model(wrapper, dataset, result_store, lock)
                all_pools.append(result_store.get(label, {}))
                wrapper.unload()
                del wrapper

        # ── MBR selection ──────────────────────────────────────────────────
        logger.info(f"Phase final — Pool merge + MBR selection")
        results: List[Tuple[str, str]] = []

        for sid in tqdm(sample_ids, desc="MBR"):
            combined: List[str] = []
            for pools in all_pools:
                combined.extend(pools.get(sid, []))

            pp     = self.postprocessor.postprocess_batch(combined) if combined else []
            chosen = self.mbr.pick(pp)

            if not chosen or not chosen.strip():
                chosen = "The tablet is too damaged to translate."

            results.append((sid, chosen))

            if len(results) % cfg.checkpoint_freq == 0:
                ckpt = Path(cfg.output_dir) / f"checkpoint_{len(results)}.csv"
                pd.DataFrame(results, columns=["id", "translation"]).to_csv(
                    ckpt, index=False
                )
                logger.info(f"  Checkpoint: {len(results)} rows → {ckpt}")

        result_df = pd.DataFrame(results, columns=["id", "translation"])
        self._validate(result_df)
        return result_df

    # ------------------------------------------------------------------

    def _validate(self, df: pd.DataFrame):
        logger = self.logger
        logger.info("=" * 60)
        empty = df["translation"].str.strip().eq("").sum()
        lens  = df["translation"].str.len()
        logger.info(f"Empty     : {empty} ({100*empty/max(1,len(df)):.2f}%)")
        logger.info(
            f"Len mean  : {lens.mean():.1f}  "
            f"median: {lens.median():.1f}  "
            f"min: {lens.min()}  max: {lens.max()}"
        )
        for idx in [0, len(df)//4, len(df)//2, 3*len(df)//4, len(df)-1]:
            row = df.iloc[idx]
            logger.info(f"  ID {row['id']}: {str(row['translation'])[:80]}")
        logger.info("=" * 60)


# ---------------------------------------------------------------------------
# Environment info
# ---------------------------------------------------------------------------

def print_env(cfg: EnsembleMBRConfig):
    print(f"PyTorch  : {torch.__version__}")
    print(f"CUDA     : {torch.cuda.is_available()}")
    n_gpus = _num_gpus()
    print(f"GPUs     : {n_gpus}")
    for i in range(n_gpus):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i} : {torch.cuda.get_device_name(i)}  {mem:.1f} GB")
    print(f"BF16 AMP   : {cfg.use_bf16_amp}")
    print(f"Multi-GPU  : {cfg.use_multi_gpu and n_gpus >= 2}")
    print(f"Models     : {len(cfg.model_paths)}")
    for i, (p, d) in enumerate(zip(cfg.model_paths, cfg.model_devices)):
        print(f"  [MadLab-{i+1}] → {d}  |  {Path(p).name}")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    cfg    = EnsembleMBRConfig()
    logger = setup_logging(cfg.output_dir)

    print_env(cfg)

    logger.info(f"Loading test data: {cfg.test_data_path}")
    test_df = pd.read_csv(cfg.test_data_path, encoding="utf-8")
    logger.info(f"Test samples: {len(test_df)}")

    engine     = EnsembleMBREngine(cfg, logger)
    results_df = engine.run(test_df)

    out_path = Path(cfg.output_dir) / "submission.csv"
    results_df.to_csv(out_path, index=False)
    logger.info(f"Saved → {out_path} ({len(results_df)} rows)")

    cfg_snap = {
        "models":             cfg.model_paths,
        "model_devices":      [str(d) for d in cfg.model_devices],
        "madlab_prefix":      MADLAB_PREFIX,
        "max_input_length":   cfg.max_input_length,
        "max_new_tokens":     cfg.max_new_tokens,
        "num_beam_cands":     cfg.num_beam_cands,
        "num_beams":          cfg.num_beams,
        "num_sample_cands":   cfg.num_sample_cands,
        "length_penalty":     cfg.length_penalty,
        "repetition_penalty": cfg.repetition_penalty,
        "mbr_top_p":          cfg.mbr_top_p,
        "mbr_temperature":    cfg.mbr_temperature,
        "mbr_pool_cap":       cfg.mbr_pool_cap,
        "use_bf16_amp":       cfg.use_bf16_amp,
        "use_multi_gpu":      cfg.use_multi_gpu,
        "batch_size":         cfg.batch_size,
    }
    with open(Path(cfg.output_dir) / "ensemble_mbr_config.json", "w") as f:
        json.dump(cfg_snap, f, indent=2)

    print("\n" + "=" * 60)
    print("Ensemble MBR complete")
    print(f"Submission : {out_path}")
    print(f"Total rows : {len(results_df)}")
    print("=" * 60)

Loading fine-tuned MADLAD-400 model from /kaggle/input/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-ciphermix-2ep/...


`torch_dtype` is deprecated! Use `dtype` instead!
2026-02-16 05:26:39.223153: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771219599.408096      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771219599.461676      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771219599.889465      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771219599.889506      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771219599.889509      55

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Model loaded successfully.
Reading /kaggle/input/deep-past-initiative-machine-translation/train.csv...
Starting translations...


Translating:   0%|          | 0/1561 [00:00<?, ?it/s]


✓ Submission saved to /kaggle/working/submission.csv
Model: /kaggle/input/datasets/ravikshdikola/madlab-akkadian-retrained-1st-stage-ciphermix-2ep/
NUM_BEAMS: 0
LENGTH_PENALTY: removed

First row check:
Translation: 'Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-...'
